ถบเเถวซ้ำ

In [ ]:
import pandas as pd

# อ่านไฟล์ Excel
df = pd.read_excel('web/data/data.xlsx')

# ตรวจสอบค่าซ้ำโดยพิจารณาคอลัมน์เหล่านี้ร่วมกัน
duplicate_rows = df[df.duplicated(subset=['มหาวิทยาลัย', 'คณะ', 'หลักสูตร', 'ลิงก์'])]

# แสดงแถวที่ซ้ำ
print("แถวที่ซ้ำกัน:")
print(duplicate_rows[['มหาวิทยาลัย', 'หลักสูตร']])

# ลบแถวที่ซ้ำ (โดยเก็บแถวแรกไว้)
df_no_duplicates = df.drop_duplicates(subset=['มหาวิทยาลัย', 'คณะ', 'หลักสูตร', 'ลิงก์'], keep='first')

# บันทึกเป็นไฟล์ใหม่
df_no_duplicates.to_excel('web/data/data_drop_duplicates.xlsx', index=False)

print("บันทึกไฟล์เรียบร้อย")


กรองเเถวออก ตามเงื่อนไข

In [ ]:
import pandas as pd

# อ่านไฟล์ Excel
df = pd.read_excel('web/data/data_drop_duplicates.xlsx')

# เงื่อนไขกรองหลักสูตรที่เกี่ยวข้อง
condition1 = (
    (df['หลักสูตร'].astype(str).str.contains('วิศวกรรม')) &
    (
        df['หลักสูตร'].astype(str).str.contains('คอมพิวเตอร์') |
        df['หลักสูตร'].astype(str).str.contains('ปัญญาประดิษฐ์')
    )
)

# เงื่อนไขกรองคณะที่ไม่ต้องการ
condition2 = ~(
    df['คณะ'].astype(str).str.contains('> วิศวกรรมทั่วไป') |
    df['หลักสูตร'].astype(str).str.contains('ครุศาสตร์') |
    df['หลักสูตร'].astype(str).str.contains('ค.อ.บ.') |
    df['หลักสูตร'].astype(str).str.contains('อส.บ.')
)

# รวมเงื่อนไขทั้งสอง
final_condition = condition1 & condition2

# กรองข้อมูล
filtered_df = df[final_condition]

# แสดงรายการที่ถูกตัดออก (optional)
dropped_df = df[~final_condition]
print("\nหลักสูตรที่ถูกตัดออก:")
for _, row in dropped_df.iterrows():
    print([row['หลักสูตร']])

# บันทึกไฟล์ใหม่
filtered_df.to_excel('web/data/filtered_data.xlsx', index=False)
print("บันทึกไฟล์เรียบร้อย")


กรองค่าเทอม (หลักสูตรที่ไม่เเสดลงค่าใช้จ่าย ต้องกรอกเอง)

In [ ]:
import pandas as pd
import re

# โหลดข้อมูล
df = pd.read_excel('web/data/filtered_data.xlsx')

# สร้างฟังก์ชันสำหรับดึงค่าเทอม
def extract_tuition(text):
    text = str(text)
    
    # ข้าม URL หรือ "ไม่พบข้อมูล"
    if 'http' in text or 'ไม่พบข้อมูล' in text:
        return None

    # ค้นหาตัวเลข เช่น 30000, 232,000, 105,000 ฯลฯ
    numbers = re.findall(r'\d{2,3}(?:,\d{3})*|\d+', text)
    numbers = [int(n.replace(',', '')) for n in numbers if int(n.replace(',', '')) >= 1000]

    # ตรวจคำว่า "ต่อภาคการศึกษา", "ต่อภาคเรียน", "ตลอดหลักสูตร"
    if 'ตลอดหลักสูตร' in text:
        term = 'ตลอดหลักสูตร'
    elif 'ต่อภาค' in text or '/เทอม' in text or '/ภาค' in text:
        term = 'ต่อภาคการศึกษา'
    else:
        term = 'ไม่ชัดเจน'

    # ส่งค่าเทอมที่มากที่สุด (มักจะเป็นค่าเทอมหลัก)
    if numbers:
        return f"{max(numbers):,} บาท {term}"
    else:
        return None

# ประมวลผลคอลัมน์ "ค่าใช้จ่าย"
df['ค่าเทอมที่ดึงได้'] = df['ค่าใช้จ่าย'].apply(extract_tuition)

# แสดงเฉพาะที่ดึงค่าเทอมได้
print(df[['มหาวิทยาลัย', 'หลักสูตร', 'ค่าเทอมที่ดึงได้']].dropna())

# บันทึกข้อมูลลงไฟล์ใหม่
df.to_excel('web/data/tuition.xlsx', index=False)

print("บันทึกไฟล์เรียบร้อย")


ค่าเทอม

In [ ]:
import pandas as pd
import re

# โหลดไฟล์ที่มีคอลัมน์ "ค่าเทอมที่ดึงได้"
df = pd.read_excel('web/data/tuition.xlsx')

# ฟังก์ชันแปลงค่าเทอมเป็นตัวเลขต่อภาค
def estimate_per_term(text):
    if pd.isna(text):
        return None

    text = str(text)

    # ค้นหาเลขจากข้อความ เช่น "120,000 บาท ตลอดหลักสูตร"
    match = re.search(r'(\d{1,3}(?:,\d{3})*|\d+)', text)
    if match:
        amount = int(match.group(1).replace(',', ''))
    else:
        return None

    # คำนวณ
    if 'ตลอดหลักสูตร' in text:
        estimated = amount // 8
    elif 'ต่อภาค' in text:
        estimated = amount
    else:
        estimated = None

    return estimated

# สร้างคอลัมน์ใหม่ "ค่าเทอม"
df['ค่าเทอม'] = df['ค่าเทอมที่ดึงได้'].apply(estimate_per_term)

# แสดงผล
# กรองเฉพาะแถวที่เป็น "ตลอดหลักสูตร"
mask = df['ค่าเทอมที่ดึงได้'].astype(str).str.contains('ตลอดหลักสูตร', na=False)
# แสดงผลเฉพาะคอลัมน์ที่ต้องการ
print(df.loc[mask, ['ค่าเทอมที่ดึงได้', 'ค่าเทอม']])


# บันทึกไฟล์ใหม่
df.to_excel('web/data/data-cleaned.xlsx', index=False)
print("บันทึกไฟล์เรียบร้อย")


ประเภทหลักสูตร

In [ ]:
import pandas as pd

# อ่านไฟล์
df = pd.read_excel("web/data/data-cleaned.xlsx")  

# ลบคอลัมน์ที่ไม่ต้องการ
df.drop(columns=["ลิงก์", "ค่าใช้จ่าย", "ค่าเทอมที่ดึงได้"], inplace=True, errors='ignore')

# ฟังก์ชันจัดหมวดหมู่ประเภทหลักสูตร
def normalize_course_type(course_type):
    course_type = str(course_type).strip()

    if "Double Degree" in course_type:
        return "นานาชาติ พิเศษ"
    elif "Joint Degree" in course_type:
        return "นานาชาติ"
    elif "นานาชาติ" in course_type:
        return "นานาชาติ"
    elif "พิเศษ" in course_type:
        return "ภาษาไทย พิเศษ"
    else:
        return "ภาษาไทย ปกติ"

# แปลงข้อมูลในคอลัมน์ 'ประเภทหลักสูตร'
df['ประเภทหลักสูตร'] = df['ประเภทหลักสูตร'].apply(normalize_course_type)

# แยก "คณะ > สาขา" → คณะหลัก และ สาขา
df[['คณะ', 'สาขา']] = df['คณะ'].str.split('>', n=1, expand=True)

# ล้างช่องว่างหัวท้าย
df['คณะ'] = df['คณะ'].str.strip()

# กรณีบางแถวไม่มี ">" จะไม่มีสาขา ต้องเติมให้เป็นค่าว่างหรือ "ไม่ระบุ"
df['สาขา'] = df['สาขา'].fillna('ไม่ระบุ').str.strip()

# กำหนดลำดับคอลัมน์ที่ต้องการใหม่นิดหน่อย
cols = list(df.columns)

# เอาคอลัมน์ 'คณะ' กับ 'สาขา' ไปอยู่ติดกันที่ตำแหน่งที่ 2 กับ 3
# สมมติคอลัมน์แรกคือ 'ประเภทหลักสูตร' อยู่ตำแหน่ง 0

# เอาคอลัมน์ 'คณะ' กับ 'สาขา' ออกก่อน (ถ้ามี)
for col in ['คณะ', 'สาขา']:
    if col in cols:
        cols.remove(col)

# แทรก 'คณะ' กับ 'สาขา' เข้าไปที่ตำแหน่ง 1 และ 2 (index เริ่ม 0)
cols.insert(2, 'คณะ')
cols.insert(3, 'สาขา')

# จัดลำดับ dataframe ตาม cols ที่แก้แล้ว
df = df[cols]


# บันทึกผลลัพธ์
df.to_excel("web/data/cleaned_file.xlsx", index=False)  

print("บันทึกไฟล์เรียบร้อย")


ภาค

In [ ]:
import pandas as pd

# โหลดจากไฟล์ Excel
df = pd.read_excel("web/data/cleaned_file.xlsx")

def get_region(university_name):
    name = university_name.lower()

    # ตรวจสอบภาคที่มีชื่อเฉพาะเจาะจงที่อาจทับซ้อนก่อน
    # ภาคตะวันออก
    if 'เกษตรศาสตร์ ศรีราชา' in name:
        return 'ตะวันออก'

    # ภาคใต้
    elif 'เจ้าคุณทหารลาดกระบัง ชุมพร' in name:
        return 'ใต้'
    elif 'ราชมงคลศรีวิชัย ตรัง' in name:
        return 'ใต้'

    # ภาคตะวันออกเฉียงเหนือ
    elif 'เกษตรศาสตร์ เฉลิมพระเกียรติ จ.สกลนคร' in name:
        return 'ตะวันออกเฉียงเหนือ'
    elif any(x in name for x in [
        'ขอนแก่น', 'อุบลราชธานี', 'เทคโนโลยีสุรนารี', 'กาฬสินธุ์',
        'ราชภัฏอุบลราชธานี', 'ราชมงคลอีสาน'
    ]):
        return 'ตะวันออกเฉียงเหนือ'

    # ภาคใต้ (ต่อจากชื่อเฉพาะด้านบน)
    elif any(x in name for x in [
        'สงขลา', 'ภูเก็ต', 'วลัยลักษณ์', 'นราธิวาสราชนครินทร์',
        'ราชมงคลศรีวิชัย'
    ]):
        return 'ใต้'

    # ภาคเหนือ
    elif any(x in name for x in [
        'เชียงใหม่', 'นเรศวร', 'แม่ฟ้าหลวง', 'พะเยา', 'ราชภัฏพิบูลสงคราม'
    ]):
        return 'เหนือ'

    # ภาคกลาง (ตรวจสอบเป็นลำดับสุดท้ายสำหรับชื่อที่อาจทับซ้อนหรือไม่เฉพาะเจาะจงมากพอ)
    elif any(x in name for x in [
        'จุฬาลงกรณ์', 'ธรรมศาสตร์', 'มหิดล', 'ศิลปากร สนามจันทร์',
        'ศรีนครินทรวิโรฒ องครักษ์', 'เกษตรศาสตร์ บางเขน', 'เกษตรศาสตร์ กำแพงแสน',
        'พระจอมเกล้าธนบุรี', 'พระจอมเกล้าพระนครเหนือ', 'เจ้าคุณทหารลาดกระบัง ลาดกระบัง',
        'หอการค้าไทย', 'มหานคร', 'ปัญญาภิวัฒน์', 'รามคำแหง', 'จิตรลดา',
        'ราชภัฏพระนคร', 'สวนสุนันทา', 'ราชมงคลธัญบุรี', 'ราชมงคลกรุงเทพ',
        'ราชมงคลพระนคร', 'ราชมงคลรัตนโกสินทร์', 'ศรีปทุม', 'รังสิต', 'ธุรกิจบัณฑิตย์',
        'เกษมบัณฑิต', 'สยาม', 'ราชมงคลรัตนโกสินทร์ ศาลายา'
    ]):
        return 'กลาง'

    return 'ไม่ทราบภาค'

# สร้างคอลัมน์ 'ภาค'
df['ภาค'] = df['มหาวิทยาลัย'].apply(get_region)

# บันทึกผลลัพธ์
df.to_excel("web/data/cleaned_with_region.xlsx", index=False)

print("บันทึกไฟล์เรียบร้อย")


พิกัด (หากหาไม่เจอ กรอกเอง)

In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# อ่านไฟล์ Excel
df = pd.read_excel("web/data/cleaned_with_region.xlsx")

# สร้าง geolocator
geolocator = Nominatim(user_agent="my_geocoder")
# ตั้ง RateLimiter เพื่อกันถูกบล็อกจากการขอข้อมูลเร็วเกินไป
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# ฟังก์ชันเพื่อตัดคำ "วิทยาเขตหลัก" ออกจากชื่อมหาวิทยาลัย (ถ้ามี)
def clean_university_name(name):
    if "วิทยาเขตหลัก" in name:
        return name.replace("วิทยาเขตหลัก", "").strip()
    return name

# ฟังก์ชันเพื่อค้นหาพิกัดจากชื่อมหาวิทยาลัย
def get_lat_lon(university_name):
    try:
        clean_name = clean_university_name(university_name)
        location = geocode(clean_name + ", Thailand")
        if location:
            return location.latitude, location.longitude
        else:
            return None, None
    except:
        return None, None

# สร้างคอลัมน์ใหม่สำหรับ Lat และ Lon
df[['Latitude', 'Longitude']] = df['มหาวิทยาลัย'].apply(lambda x: pd.Series(get_lat_lon(x)))

# บันทึกเป็นไฟล์ใหม่
df.to_excel("web/data/university_fee_with_latlon1.xlsx", index=False)

print(df[['มหาวิทยาลัย', 'Latitude', 'Longitude']])
print("บันทึกไฟล์เรียบร้อย")
